# 05 — Build `data/pbmc_subset.h5ad` and write DATA.md
Recipe (deterministic):
* **donors**: `Donor8` + the 3 other donors with the most PBS (control) cells
* **cytokines (20)**: `IL-15` + every other cytokine in its family (`Common γ chain/IL-13/TSLP`), then filled up to 20
  with a seeded random draw (`numpy.random.default_rng(0)`, from the sorted list of remaining non-PBS cytokines);
  plus `PBS` controls for every retained donor
* **all cell types**; `.uns` (incl. `esm2_embeddings`, `donor_embeddings`) kept as-is

In [1]:
import os
DATA = "/fast/AG_Bunina/Pavel_1/data/cellflow"
ROOT = "/fast/AG_Bunina/Pavel_1/cellflow-regen"
os.environ["HF_HOME"] = f"{DATA}/hf_cache"
import hashlib, time, numpy as np, pandas as pd, anndata as ad
from importlib.metadata import version
from cellflow.datasets import pbmc_cytokines
adata = pbmc_cytokines(backed="r")        # from the /fast HF cache; X stays on disk
obs = adata.obs[["donor", "cytokine", "cell_type", "cytokine_family"]].copy()
SRC = adata.filename
print(SRC, adata.shape)

/fast/AG_Bunina/Pavel_1/miniforge3/envs/cellflow/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache/hub/datasets--theislab--cellflow-datasets/snapshots/4b235c9a1e0073c95211c6ef6257db184fc83f71/pbmc_parse.h5ad (9697974, 2000)


In [2]:
# donors: Donor8 + top-3 other donors by PBS cell count (ties broken by name)
pbs_per_donor = obs.loc[obs.cytokine == "PBS", "donor"].value_counts()
pbs_per_donor = pbs_per_donor.rename_axis("donor").reset_index(name="n").sort_values(["n", "donor"], ascending=[False, True])
print(pbs_per_donor.to_string(index=False))
DONORS = ["Donor8"] + [d for d in pbs_per_donor.donor if d != "Donor8"][:3]
print("DONORS =", DONORS)

  donor     n
 Donor1 90452
Donor12 71179
 Donor2 70520
 Donor6 59158
 Donor5 55293
 Donor3 48567
 Donor4 44518
Donor11 43955
 Donor8 43064
 Donor7 37816
Donor10 35850
 Donor9 29329
DONORS = ['Donor8', 'Donor1', 'Donor12', 'Donor2']


In [3]:
# cytokines: IL-15 + its family, then seeded fill to 20
fam = obs.drop_duplicates("cytokine").set_index("cytokine")["cytokine_family"].astype(str)
il15_family = fam["IL-15"]
same_family = sorted(c for c in fam.index if fam[c] == il15_family and c not in ("IL-15", "PBS"))
print(f"IL-15 family '{il15_family}':", same_family)
CYTOKINES = ["IL-15"] + same_family
pool = sorted(c for c in fam.index if c not in CYTOKINES and c != "PBS")
rng = np.random.default_rng(0)
CYTOKINES += sorted(rng.choice(pool, size=20 - len(CYTOKINES), replace=False).tolist())
assert len(CYTOKINES) == 20 and len(set(CYTOKINES)) == 20 and "IL-15" in CYTOKINES
print("CYTOKINES =", CYTOKINES)

IL-15 family 'Common γ chain/IL-13/TSLP': ['IL-13', 'IL-2', 'IL-21', 'IL-4', 'IL-7', 'IL-9', 'TSLP']
CYTOKINES = ['IL-15', 'IL-13', 'IL-2', 'IL-21', 'IL-4', 'IL-7', 'IL-9', 'TSLP', 'ADSF', 'BAFF', 'C5a', 'FGF-beta', 'GM-CSF', 'IFN-epsilon', 'IL-17B', 'IL-20', 'IL-31', 'IL-6', 'LT-alpha2-beta1', 'SCF']


In [4]:
mask = obs.donor.isin(DONORS).to_numpy() & obs.cytokine.isin(CYTOKINES + ["PBS"]).to_numpy()
idx = np.flatnonzero(mask)
print(f"selected cells: {idx.size:,}")
t = time.time()
sub = adata[idx].to_memory()
print(f"materialized in {time.time()-t:.0f}s:", sub)
for c in ["donor", "cytokine", "cell_type", "cytokine_family"]:
    sub.obs[c] = sub.obs[c].cat.remove_unused_categories()

selected cells: 1,114,081


materialized in 64s: AnnData object with n_obs × n_vars = 1114081 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'cytokine_family'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'donor_embeddings', 'esm2_embeddings', 'hvg', 'log1p'


In [5]:
# sanity checks: PBS and IL-15 for every donor, all 20 cytokines x 4 donors non-empty, embeddings present
ct = pd.crosstab(sub.obs.cytokine, sub.obs.donor)
missing = [(c, d) for c in CYTOKINES + ["PBS"] for d in DONORS if ct.loc[c, d] == 0]
assert all(ct.loc["PBS", d] > 0 for d in DONORS), "PBS missing for a donor — test split impossible"
assert all(ct.loc["IL-15", d] > 0 for d in DONORS), "IL-15 missing for a donor"
print("PBS per donor:", ct.loc["PBS"].to_dict())
print("IL-15 per donor:", ct.loc["IL-15"].to_dict())
print("empty (cytokine, donor) cells:", missing)
assert all(c in sub.uns["esm2_embeddings"] for c in CYTOKINES), "cytokine without ESM2 embedding"
assert all(d in sub.uns["donor_embeddings"] for d in DONORS)
print("n cell types:", sub.obs.cell_type.nunique(), "of", obs.cell_type.nunique())

PBS per donor: {'Donor1': 90452, 'Donor2': 70520, 'Donor8': 43064, 'Donor12': 71179}
IL-15 per donor: {'Donor1': 9863, 'Donor2': 9823, 'Donor8': 8939, 'Donor12': 13166}
empty (cytokine, donor) cells: []
n cell types: 18 of 18


In [6]:
OUT = f"{DATA}/pbmc_subset.h5ad"
sub.uns["subset_recipe"] = {"donors": DONORS, "cytokines": CYTOKINES, "controls": "PBS", "seed": 0,
                            "source": "cellflow.datasets.pbmc_cytokines() / pbmc_parse.h5ad"}
t = time.time()
sub.write_h5ad(OUT, compression="gzip")
print(f"written in {time.time()-t:.0f}s")
h = hashlib.sha256()
with open(OUT, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
SHA, SIZE = h.hexdigest(), os.path.getsize(OUT)
print(OUT, f"{SIZE:,} bytes ({SIZE/1e9:.2f} GB)", SHA)

written in 16s


/fast/AG_Bunina/Pavel_1/data/cellflow/pbmc_subset.h5ad 331,850,186 bytes (0.33 GB) 61883f332841d2af7becd91d4d650d7ef24f329e5f0b33df6dd8ca617509edc1


In [7]:
# round-trip check
chk = ad.read_h5ad(OUT, backed="r")
assert chk.shape == sub.shape and list(chk.uns["subset_recipe"]["donors"]) == DONORS
print("re-read OK:", chk.shape)

re-read OK: (1114081, 2000)


In [8]:
# write DATA.md
def md_table(df, index=True):
    df = df.reset_index() if index else df
    rows = [list(map(str, df.columns))] + [[f"{v:,}" if isinstance(v, (int, np.integer)) else str(v) for v in r] for r in df.itertuples(index=False)]
    return "\n".join(["| " + " | ".join(r) + " |" for r in rows[:1]] + ["|" + "---|" * len(rows[0])] + ["| " + " | ".join(r) + " |" for r in rows[1:]])
per_donor = sub.obs.donor.value_counts().reindex(DONORS)
per_cyt = sub.obs.cytokine.value_counts().reindex(CYTOKINES + ["PBS"])
ct_full = ct.reindex(index=CYTOKINES + ["PBS"], columns=DONORS)
ct_full["total"] = ct_full.sum(axis=1)
md = f"""# DATA — PBMC cytokine subset

**File:** `{OUT}` (also reachable as `{ROOT}/data/pbmc_subset.h5ad`)
**Size:** {SIZE:,} bytes ({SIZE/1e9:.2f} GB) · **sha256:** `{SHA}`
**Shape:** {sub.n_obs:,} cells × {sub.n_vars:,} genes · all {sub.obs.cell_type.nunique()} cell types · written with `compression="gzip"`
**Built by:** [`notebooks/05_build_subset.ipynb`](notebooks/05_build_subset.ipynb) (kernel `cellflow`)

## Source
* `cellflow.datasets.pbmc_cytokines()` (cellflow-tools {version('cellflow-tools')}, anndata {version('anndata')}) →
  HF dataset `theislab/cellflow-datasets`, file `pbmc_parse.h5ad`, revision `4b235c9a1e0073c95211c6ef6257db184fc83f71`
* 13,101,592,422 bytes, LFS sha256 `c72671b36248b82c559211a28fbf009f4f077e3b2097f9ba6a04d27d27edd4c9`
* 9,697,974 cells × 2,000 HVGs; cached at `{SRC}` (`HF_HOME={DATA}/hf_cache`)

## Recipe
1. **Donors (4):** `Donor8` + the 3 other donors with the most PBS cells in the full data (ties by name)
   → `{DONORS}`
2. **Cytokines (20):** `IL-15` + all other cytokines in its family (`{il15_family}`): `{same_family}`;
   remaining {20 - 1 - len(same_family)} slots drawn with `np.random.default_rng(0).choice(pool, size={20 - 1 - len(same_family)}, replace=False)`
   where `pool` = sorted non-PBS cytokines not already chosen
   → `{CYTOKINES}`
3. **Controls:** `PBS` for all 4 donors.
4. **Cells:** `donor ∈ DONORS and cytokine ∈ CYTOKINES ∪ {{PBS}}`, all cell types. `obs`, `var`, `uns` kept
   (unused categories removed); recipe stored in `uns['subset_recipe']`.

Checks passed: PBS and IL-15 present for every donor; all 20 cytokines have an entry in `uns['esm2_embeddings']`
(2560-d ESM2 vectors); all donors in `uns['donor_embeddings']`.

## Cells per donor
{md_table(per_donor.rename_axis('donor').reset_index(name='cells'), index=False)}

## Cells per cytokine
{md_table(per_cyt.rename_axis('cytokine').reset_index(name='cells'), index=False)}

## Cells per cytokine × donor
{md_table(ct_full.rename_axis('cytokine'))}

## Regenerate
```bash
source /fast/AG_Bunina/Pavel_1/miniforge3/bin/activate /fast/AG_Bunina/Pavel_1/miniforge3/envs/cellflow
cd {ROOT}/notebooks
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=cellflow 05_build_subset.ipynb
sha256sum {OUT}   # should match the value above
```
Environment: see `{ROOT}/requirements.lock.txt` and `notebooks/02_env_setup.ipynb`.
"""
open(f"{ROOT}/DATA.md", "w").write(md)
print(md[:1500])

# DATA — PBMC cytokine subset

**File:** `/fast/AG_Bunina/Pavel_1/data/cellflow/pbmc_subset.h5ad` (also reachable as `/fast/AG_Bunina/Pavel_1/cellflow-regen/data/pbmc_subset.h5ad`)
**Size:** 331,850,186 bytes (0.33 GB) · **sha256:** `61883f332841d2af7becd91d4d650d7ef24f329e5f0b33df6dd8ca617509edc1`
**Shape:** 1,114,081 cells × 2,000 genes · all 18 cell types · written with `compression="gzip"`
**Built by:** [`notebooks/05_build_subset.ipynb`](notebooks/05_build_subset.ipynb) (kernel `cellflow`)

## Source
* `cellflow.datasets.pbmc_cytokines()` (cellflow-tools 0.0.9, anndata 0.12.19) →
  HF dataset `theislab/cellflow-datasets`, file `pbmc_parse.h5ad`, revision `4b235c9a1e0073c95211c6ef6257db184fc83f71`
* 13,101,592,422 bytes, LFS sha256 `c72671b36248b82c559211a28fbf009f4f077e3b2097f9ba6a04d27d27edd4c9`
* 9,697,974 cells × 2,000 HVGs; cached at `/fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache/hub/datasets--theislab--cellflow-datasets/snapshots/4b235c9a1e0073c95211c6ef6257db184fc83f71/pbmc